In [49]:
from data_model_loader import *

model = load_model()
annotations, images = load_coco_2014_dataset()

with open('config.json', 'r') as file:
    config = json.load(file)

coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']

OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Get Annotations ..
Randomly selecting 1500 pictures ..
Done!


In [61]:
from PIL import Image
import torchvision.transforms as transforms
import pathlib
from tqdm import tqdm


def predict(model, data_path):
    data_path = pathlib.Path(data_path)
    predictions = {}

    for img in tqdm(images[:10], desc="Loading images"):
        # load image
        image_path = data_path/"val2014"/"val2014"/img
        image = Image.open(image_path)

        # transform to tensor
        transform = transforms.Compose([
            transforms.ToTensor()  # Konvertiert das Bild zu einem Tensor [C, H, W] mit Werten zwischen 0 und 1
        ])
        image_tensor = transform(image)
        image_tensor = (image_tensor * 255).byte()  # convert to uint8
        image_tensor = image_tensor.permute(1, 2, 0)  # [C, H, W] -> [H, W, C]
        image_tensor = image_tensor.unsqueeze(0)

        # inference
        detector_output = model(image_tensor)
        score = detector_output["detection_scores"].numpy()[0][0]
        detected_class = detector_output["detection_classes"].numpy()[0][0]
        
        # eval
        predictions[img] = {
            "class": detected_class,
            "score": score
        }
        

    return predictions
    
def get_annotation(annotation_file_path):
    with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)
    category_map = {category['id']: category['name'] for category in coco_data['categories']}
    return category_map


In [62]:
results = predict(model, coco_folder)
results

Loading images: 100%|██████████| 10/10 [00:00<00:00, 24.45it/s]


{'COCO_val2014_000000518721.jpg': {'class': 25.0, 'score': 0.77734715},
 'COCO_val2014_000000142483.jpg': {'class': 59.0, 'score': 0.7786198},
 'COCO_val2014_000000511204.jpg': {'class': 22.0, 'score': 0.86067975},
 'COCO_val2014_000000000962.jpg': {'class': 43.0, 'score': 0.7884087},
 'COCO_val2014_000000501420.jpg': {'class': 1.0, 'score': 0.73010594},
 'COCO_val2014_000000224004.jpg': {'class': 75.0, 'score': 0.8060202},
 'COCO_val2014_000000574509.jpg': {'class': 48.0, 'score': 0.72362936},
 'COCO_val2014_000000559550.jpg': {'class': 1.0, 'score': 0.8177213},
 'COCO_val2014_000000182984.jpg': {'class': 20.0, 'score': 0.7552063},
 'COCO_val2014_000000142698.jpg': {'class': 52.0, 'score': 0.6934609}}

In [63]:
classes = get_annotation(annotation_file_path)

In [64]:
for i in results.keys():
    print(classes[results[i]["class"]], results[i]["score"]) # annotations[results[i]["class"]]

giraffe 0.77734715
pizza 0.7786198
elephant 0.86067975
tennis racket 0.7884087
person 0.73010594
remote 0.8060202
fork 0.72362936
person 0.8177213
sheep 0.7552063
banana 0.6934609
